# 04 — Performance & Evaluation

Two critical production concerns: making your LLM calls faster/cheaper with caching, and programmatically evaluating the quality of your RAG pipeline.

## Setup

In [23]:
import os
import time
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_qdrant import QdrantVectorStore

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)
embed_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", task_type="retrieval_document")

print("Models ready")

Models ready


## Step 17 — LLM Caching

Every identical LLM call wastes tokens and time. LangChain's `InMemoryCache` intercepts duplicate queries and returns cached results instantly.

In [24]:
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

# Init semantic cache vector store (chat history)
cache_client = QdrantClient(":memory:")
cache_client.create_collection(
    collection_name="chat_history",
    vectors_config=VectorParams(size=3072, distance=Distance.COSINE),
)
cache_store = QdrantVectorStore(
    client=cache_client,
    collection_name="chat_history",
    embedding=embed_model,
)

THRESHOLD = 0.80
cache_answers = {}  # question -> answer

In [28]:
def semantic_cache(query: str):
    results = cache_store.similarity_search_with_score(query, k=1)
    if results and results[0][1] >= THRESHOLD:
        past_q, score = results[0]
        cached = cache_answers.get(past_q.page_content)
        if cached:
            print(f"  → Cache hit (similarity: {score:.3f})")
            return cached
    return None

def ask_with_cache(query: str):
    cached = semantic_cache(query)
    if cached:
        return cached
    print(f"  → Cache miss, invoking LLM...")
    answer = llm.invoke(query).text
    cache_store.add_texts([query])
    cache_answers[query] = answer
    return answer

In [29]:
# Demo: two semantically similar queries
q1 = "What is the capital of Indonesia?"
q2 = "Ibu kota Indonesia di mana?"

# First call — cache miss, hits API
print("Asking first time..")
result1 = ask_with_cache(q1)
print(f"{q1}: {result1}")

# Second call — semantically similar, cache hit
print("Asking second time..")
result2 = ask_with_cache(q2)
print(f"{q1}: {result2}")


Asking first time..
  → Cache hit (similarity: 1.000)
What is the capital of Indonesia?: The capital of Indonesia is **Jakarta**. 

However, the Indonesian government is currently in the process of moving the capital to a new city called **Nusantara**, located on the island of Borneo. While the transition is underway, Jakarta remains the official capital for the time being.
Asking second time..
  → Cache hit (similarity: 0.931)
What is the capital of Indonesia?: The capital of Indonesia is **Jakarta**. 

However, the Indonesian government is currently in the process of moving the capital to a new city called **Nusantara**, located on the island of Borneo. While the transition is underway, Jakarta remains the official capital for the time being.


In [35]:
# Langfuse — LLM tracing dashboard
# Sign up free at https://cloud.langfuse.com, then add to .env:
#   LANGFUSE_PUBLIC_KEY=...
#   LANGFUSE_SECRET_KEY=...
#   LANGFUSE_HOST=https://cloud.langfuse.com  (optional, for self-host)

from langfuse.langchain import CallbackHandler

langfuse_handler = CallbackHandler()

response = llm.invoke(
    "What is the capital of Indonesia?",
    config={"callbacks": [langfuse_handler]}
)


# Flush ensures traces are sent synchronously before the cell ends
langfuse_handler._langfuse_client.flush()

print(response.text[:100] + "...")
print("\n\u2705 Trace sent to Langfuse! Check your dashboard.")

The capital of Indonesia is **Jakarta**. 

However, the Indonesian government is currently in the pr...

✅ Trace sent to Langfuse! Check your dashboard.


Failed to export span batch code: 401, reason: Unauthorized


### Custom Callbacks

LangChain also supports **custom callbacks** — subclass `BaseCallbackHandler` and override lifecycle methods like `on_llm_start`, `on_llm_end`, `on_chain_start`. This lets you plug in logging, metrics, or any custom logic into every LLM invocation.

In [34]:
from langchain_core.callbacks import BaseCallbackHandler

class PrintCallback(BaseCallbackHandler):
    def on_llm_start(self, serialized, prompts, **kwargs):
        print(f"\u23f3 LLM receiving: {prompts[0][:60]}...")
    def on_llm_end(self, response, **kwargs):
        t = response.generations[0][0].text
        print(f"\u2705 LLM replied: {t[:60]}...")

# Multiple callbacks can run together
response = llm.invoke(
    "What is the capital of Indonesia?",
    config={"callbacks": [PrintCallback()]}
)
print(f"\nFull response: {response.text[:100]}")

⏳ LLM receiving: Human: What is the capital of Indonesia?...
✅ LLM replied: The capital of Indonesia is **Jakarta**. 

However, the Indo...

Full response: The capital of Indonesia is **Jakarta**. 

However, the Indonesian government is currently in the pr
